# PneumoNet: On-Device Continual Learning for Point-of-Care Pneumonia Diagnosis

This notebook reproduces the main experimental workflow for **PneumoNet**, a lightweight domain-incremental continual learning framework for pneumonia detection from chest X-rays.

The notebook includes:

- domain-shifted PneumoniaMNIST construction
- lightweight CNN and baseline model definitions
- replay buffer implementations
- dynamic class-weighted continual learning
- accuracy, forgetting, and efficiency evaluation

> Recommended runtime: Google Colab with GPU.


## 1. Install Dependencies

Install MedMNIST, which provides the PneumoniaMNIST dataset used in this notebook.

In [ ]:
!pip install -q medmnist


## 2. Environment Setup

Mount Google Drive, import libraries, and define the dataset storage path.

In [ ]:
from google.colab import drive
import os

# Clear any previous Google Drive mount state before mounting.
try:
    drive.flush_and_unmount()
    print("Drive unmounted successfully.")
except ValueError:
    print("Drive was not mounted, proceeding with mount.")

# Mount Google Drive to store or reuse downloaded MedMNIST data.
drive.mount('/content/drive')

# Dataset root used by MedMNIST. Change this path if you prefer a different storage location.
dataset_root = '/content/drive/MyDrive/Colab/data/MedMNIST'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
import medmnist
from medmnist import INFO
import matplotlib.pyplot as plt
import random
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

## 3. Reproducibility and Dataset Metadata

Set random seeds, select the compute device, and load PneumoniaMNIST metadata from MedMNIST.

In [ ]:
# Reproducibility and device setup
torch.manual_seed(0)
random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_flag = 'pneumoniamnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

## 4. Domain-Shifted PneumoniaMNIST

The original PneumoniaMNIST dataset is converted into five sequential domains:

1. **Base**: original PneumoniaMNIST images  
2. **LowDose**: reduced intensity and noise injection  
3. **Portable**: uneven illumination and blur  
4. **Anatomical**: translation and scaling  
5. **Institutional**: brightness, contrast, and sharpness variation


In [ ]:
# Domain-shift transformations
transform_base = transforms.ToTensor()

# Each transform simulates a clinically motivated domain shift.
class LowDoseTransform:
    """
    Simulates low-dose X-ray acquisition by reducing intensity and adding Poisson-like noise.
    """
    def __init__(self, scale_factor=0.5, noise_level=0.1):
        self.scale_factor = scale_factor
        self.noise_level = noise_level

    def __call__(self, img):
        # The transform expects tensors because it is applied after transforms.ToTensor().
        if not isinstance(img, torch.Tensor):
             raise TypeError("Input to LowDoseTransform must be a PyTorch Tensor.")

        img = img * self.scale_factor
        # Add a small epsilon to avoid division by zero when computing Poisson noise.
        noise = torch.poisson(img / (self.noise_level + 1e-6)) * self.noise_level
        img = img + noise
        img = torch.clamp(img, 0, 1)  # Ensure values are within a valid range
        return img

class PortableTransform:
    """
    Simulates portable ICU X-ray acquisition using non-uniform illumination and mild blur.
    """
    def __init__(self, gradient_strength=0.1, blur_radius=1):
        self.gradient_strength = gradient_strength
        self.blur_radius = blur_radius

    def __call__(self, img):
        # Add a subtle left-to-right intensity gradient to mimic uneven illumination.
        height, width = img.shape[-2:]
        gradient = torch.linspace(0, self.gradient_strength, width).unsqueeze(0).repeat(height, 1)
        if img.ndim == 3:
            gradient = gradient.unsqueeze(0) # Add channel dimension if needed
        img = img + gradient
        img = torch.clamp(img, 0, 1)

        # Apply mild blur to mimic loss of sharpness in portable imaging.
        if self.blur_radius > 0:
            avg_pool = nn.AvgPool2d(kernel_size=self.blur_radius*2+1, stride=1, padding=self.blur_radius)
            img = avg_pool(img.unsqueeze(0)).squeeze(0)

        return img

class AnatomicalShiftTransform:
    """
    Simulates anatomical variation through small translations and scaling changes.
    """
    def __init__(self, max_translation=2, max_scale_factor=0.1, output_size=(28, 28)):
        self.max_translation = max_translation
        self.max_scale_factor = max_scale_factor
        self.output_size = output_size

    def __call__(self, img):
        # Apply random translation to mimic differences in patient positioning.
        dx = random.randint(-self.max_translation, self.max_translation)
        dy = random.randint(-self.max_translation, self.max_translation)
        img = torch.roll(img, shifts=(dy, dx), dims=(-2, -1))

        # Apply random scaling to mimic body-habitus or posture variation.
        scale_factor = 1.0 + random.uniform(-self.max_scale_factor, self.max_scale_factor)
        if scale_factor != 1.0:
            # Resize while preserving the channel dimension.
            if img.ndim == 3:
                c, h, w = img.shape
                new_h, new_w = int(h * scale_factor), int(w * scale_factor)
                # torchvision resize supports tensor input.
                img = transforms.functional.resize(img, (new_h, new_w))

        # Center crop to preserve the original 28 x 28 PneumoniaMNIST resolution.
        img = transforms.functional.center_crop(img, self.output_size)


        return img

class InstitutionalShiftTransform:
    """
    Simulates inter-institutional variation using brightness, contrast, and sharpness changes.
    """
    def __init__(self, brightness_factor=0.2, contrast_factor=0.2, sharpness_factor=0.2):
        self.brightness_factor = brightness_factor
        self.contrast_factor = contrast_factor
        self.sharpness_factor = sharpness_factor

    def __call__(self, img):
        # Apply random brightness, contrast, and sharpness adjustments.
        img = transforms.functional.adjust_brightness(img, 1.0 + random.uniform(-self.brightness_factor, self.brightness_factor))
        img = transforms.functional.adjust_contrast(img, 1.0 + random.uniform(-self.contrast_factor, self.contrast_factor))
        # Approximate sharpness adjustment by adding a scaled high-frequency residual.
        if self.sharpness_factor > 0:
            blurred_img = transforms.functional.gaussian_blur(img, kernel_size=3)
            img = img + (img - blurred_img) * self.sharpness_factor

        img = torch.clamp(img, 0, 1)
        return img

domain_names = ["Base", "LowDose", "Portable", "Anatomical", "Institutional"]
domain_transforms = [
    transforms.ToTensor(),  # Base domain: original image tensor
    transforms.Compose([transforms.ToTensor(), LowDoseTransform()]),
    transforms.Compose([transforms.ToTensor(), PortableTransform()]),
    transforms.Compose([transforms.ToTensor(), AnatomicalShiftTransform()]),
    transforms.Compose([transforms.ToTensor(), InstitutionalShiftTransform()])
]

## 5. Model Definitions

This section defines the model architectures used in the experiments.

The paper-default PneumoNet model is implemented as `Tiny_64_CNN`, a compact CNN with two convolutional-pooling blocks and a small classifier. Additional baselines include an MLP, a larger CNN, and MobileNetV2 variants.


In [ ]:
from torchvision import models
from torchvision.models import MobileNet_V2_Weights

# Baseline model 1: Multi-layer perceptron
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)   # Binary class
        )

    def forward(self, x):
        return self.layers(x)

# Baseline model 2: Standard CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# PneumoNet CNN: lightweight architecture used for the main experiments
class Tiny_64_CNN(nn.Module):
    """Lightweight PneumoNet CNN used in the paper-default setting."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 5 * 5, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

# Smaller Tiny CNN variant for optional ablation
class Tiny_32_CNN(nn.Module):
    """Smaller CNN variant for quick ablations or extremely constrained settings."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, 1),    # (1, 28, 28) → (8, 26, 26)
            nn.ReLU(),
            nn.MaxPool2d(2),          # (8, 26, 26) → (8, 13, 13)

            nn.Conv2d(8, 16, 3, 1),   # (8, 13, 13) → (16, 11, 11)
            nn.ReLU(),
            nn.MaxPool2d(2),          # (16, 11, 11) → (16, 5, 5)

            nn.Flatten(),             # 16 × 5 × 5 = 400
            nn.Linear(400, 32),       # much smaller hidden layer
            nn.ReLU(),
            nn.Linear(32, 2)          # binary classification
        )

    def forward(self, x):
        return self.net(x)

# Baseline model 3: MobileNetV2 trained from scratch
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = models.mobilenet_v2(weights=None)  # no pretrained weights
        self.model.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        self.model.classifier[1] = nn.Linear(self.model.last_channel, num_classes)

    def forward(self, x):
        return self.model(x)


class MobileNetV2_Pretrained(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Load ImageNet-pretrained MobileNetV2.
        weights = MobileNet_V2_Weights.IMAGENET1K_V1
        self.model = models.mobilenet_v2(weights=weights)

        # Modify the first convolution so the model accepts 1-channel X-ray images.
        old_conv = self.model.features[0][0]
        self.model.features[0][0] = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None,
        )

        # Initialize grayscale weights by averaging pretrained RGB filters.
        with torch.no_grad():
            self.model.features[0][0].weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

        # Replace the ImageNet classifier with a binary normal-versus-pneumonia classifier.
        self.model.classifier[1] = nn.Linear(self.model.last_channel, num_classes)

    def forward(self, x):
        return self.model(x)


## 6. Replay Buffers

The notebook compares three replay-buffer strategies:

- **ReservoirBuffer**: standard experience replay with reservoir sampling  
- **CBRSBuffer**: class-balanced reservoir sampling  
- **DualStageBuffer**: PneumoNet buffer with class-balanced storage and class-aware replay sampling  

The proposed dual-stage buffer is designed to reduce majority-class dominance under class-imbalanced domain streams.


In [ ]:
# Replay buffer implementations

# Buffer 1: classic reservoir replay buffer
class ReservoirBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.n_seen = 0

    # Add new samples using standard reservoir sampling.
    def add_batch(self, images, labels):
        for x, y in zip(images, labels):
            self.n_seen += 1
            if len(self.buffer) < self.capacity:
                self.buffer.append((x.cpu(), y.cpu()))
            else:
                idx = random.randint(0, self.n_seen - 1)
                if idx < self.capacity:
                    self.buffer[idx] = (x.cpu(), y.cpu())

    # Sample a random replay batch from the stored examples.
    def sample(self, batch_size):
        if len(self.buffer) == 0:
            return None, None
        samples = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        imgs, labels = zip(*samples)
        return torch.stack(imgs), torch.tensor(labels)

# Buffer 2: class-balanced reservoir sampling buffer
class CBRSBuffer:
    def __init__(self, capacity_per_class=5, num_classes=10):
        self.capacity_per_class = capacity_per_class
        self.num_classes = num_classes
        self.buffer = {cls: [] for cls in range(num_classes)}
        self.class_seen_count = {cls: 0 for cls in range(num_classes)}

    def add_batch(self, images, labels):
        for x, y in zip(images, labels):
            cls = y.item()
            self.class_seen_count[cls] += 1
            seen = self.class_seen_count[cls]
            #print(f"[CBRS] Seen class {cls}: {seen} samples")
            if len(self.buffer[cls]) < self.capacity_per_class:
                self.buffer[cls].append((x.cpu(), y.cpu()))
                #print(f"[CBRS] -> Added to class {cls} buffer (size now {len(self.buffer[cls])})")
            else:
                prob = self.capacity_per_class / seen
                rand_val = random.random()
                if rand_val < prob:
                    idx = random.randint(0, self.capacity_per_class - 1)
                    self.buffer[cls][idx] = (x.cpu(), y.cpu())
                    #print(f"[CBRS] -> Replaced entry in class {cls} buffer (rand={rand_val:.3f} < prob={prob:.3f})")
                #else:
                    #print(f"[CBRS] -> Skipped (rand={rand_val:.3f} >= prob={prob:.3f})")

    # Sample uniformly across all stored examples.
    def sample(self, batch_size):
      all_samples = [sample for samples in self.buffer.values() for sample in samples]
      if not all_samples:
          return None, None

      selected = random.sample(all_samples, min(batch_size, len(all_samples)))
      x, y = zip(*selected)
      return torch.stack(x), torch.tensor(y)

from collections import defaultdict, Counter

# Buffer 3: dual-stage balanced buffer
# Stage 1: class-specific reservoir storage.
# Stage 2: class-aware balanced replay sampling.
class DualStageBuffer:
    def __init__(self, capacity_per_class=5, num_classes=10):
        self.capacity_per_class = capacity_per_class
        self.num_classes = num_classes
        self.buffer = {cls: [] for cls in range(num_classes)}
        self.class_seen_count = {cls: 0 for cls in range(num_classes)}

    # Stage 1: store samples through class-specific reservoir sampling.
    def add_batch(self, images, labels):
        for x, y in zip(images, labels):
            cls = y.item()
            self.class_seen_count[cls] += 1
            seen = self.class_seen_count[cls]
            if len(self.buffer[cls]) < self.capacity_per_class:
                self.buffer[cls].append((x.cpu(), y.cpu()))
            else:
                prob = self.capacity_per_class / seen
                if random.random() < prob:
                    idx = random.randint(0, self.capacity_per_class - 1)
                    self.buffer[cls][idx] = (x.cpu(), y.cpu())

    # Stage 2: sample replay examples as evenly as possible across classes.
    def sample(self, batch_size):
        x_list, y_list = [], []
        available_classes = [cls for cls in self.buffer if self.buffer[cls]]
        if not available_classes:
            return None, None

        samples_per_class = max(1, batch_size // len(available_classes))
        for cls in available_classes:
            class_samples = self.buffer[cls]
            k = min(samples_per_class, len(class_samples))
            selected = random.sample(class_samples, k)
            x_part, y_part = zip(*selected)
            x_list.extend(x_part)
            y_list.extend(y_part)

        total = len(x_list)
        if total < batch_size:
            leftovers = [s for cls in self.buffer.values() for s in cls]
            selected = random.sample(leftovers, min(batch_size - total, len(leftovers)))
            x_part, y_part = zip(*selected)
            x_list.extend(x_part)
            y_list.extend(y_part)

        return torch.stack(x_list), torch.tensor(y_list)

## 7. Training and Evaluation Utilities

This section defines:

- baseline training with experience replay
- PneumoNet training with dual-stage replay and dynamic class-weighted loss
- joint-training / fine-tuning training function
- evaluation function for domain-wise accuracy


In [ ]:
# Training function for experience replay baselines
def train_with_er(model, loader, buffer):
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for x, y in loop:
            x = x.to(device)  # current-domain images
            y = y.squeeze().long().to(device)  # current-domain labels

            # Retrieve replay samples from the buffer.
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))
            if x_replay is not None:
                x = torch.cat([x, x_replay.to(device)])
                y = torch.cat([y, y_replay.to(device)])

            # Update the model on the combined current + replay batch.
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()

            # Store only the current-domain samples in the replay buffer.
            buffer.add_batch(x[:batch_size].detach().cpu(), y[:batch_size].detach().cpu())

            # Show current loss in the progress bar.
            loop.set_postfix(loss=loss.item())

# Online class-frequency tracker for dynamic class-weighted loss
class ClassFrequencyTracker:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.counts = torch.zeros(num_classes)
        self.total_seen = 0

    # Update internal class counts as new labels are observed.
    def update(self, labels):
        for label in labels:
            self.counts[label.item()] += 1
        self.total_seen += len(labels)

    # Return inverse-frequency class weights.
    def get_weights(self, normalize=True):
        epsilon = 1e-6
        freqs = self.counts.clone()
        freqs[freqs == 0] = epsilon
        inv_freqs = self.total_seen / freqs
        if normalize:
            inv_freqs = inv_freqs / inv_freqs.sum() * self.num_classes
        return inv_freqs

def train_with_er_dynamic_weight(model, loader, buffer):
    model.train()
    # Paper-default optimizer: Adam.
    # To run the optimizer ablation, replace Adam with SGD.
    opt = optim.Adam(model.parameters(), lr=lr)

    tracker = ClassFrequencyTracker(num_classes=2)

    for epoch in range(num_epochs):
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0
        for x, y_new in loop:
            x = x.to(device)  # current-domain images
            y_new = y_new.squeeze().long().to(device)  # current-domain labels

            # Retrieve replay samples from the buffer.
            x_replay, y_replay = buffer.sample(int(replay_ratio * x.size(0)))

            if x_replay is not None:
                x_combined = torch.cat([x, x_replay.to(device)])
                y_combined = torch.cat([y_new, y_replay.to(device)])
            else:
                x_combined = x
                y_combined = y_new

            # Update dynamic class weights using the combined current + replay batch.
            tracker.update(y_combined)
            weights = tracker.get_weights().to(device)
            loss_fn = nn.CrossEntropyLoss(weight=weights)
            #################################################

            # Update the model on the combined current + replay batch.
            opt.zero_grad()
            loss = loss_fn(model(x_combined), y_combined)
            loss.backward()
            opt.step()

            # Store only the current-domain samples in the replay buffer. (add the new batch to the buffer)
            buffer.add_batch(x.detach().cpu(), y_new.detach().cpu())

            # Accumulate loss for epoch-level reporting.
            total_loss += loss.item()
        #print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}, Diversity = {diversity:.4f}")
        print(f"Epoch {epoch+1}: Loss = {total_loss / len(loader):.4f}")

        # Removed the print statements for accumulated class counts and weights
        #print(f"[Epoch {epoch+1}] Current weights: {weights.tolist()}")


# Training function for joint training and fine-tuning baselines
def train_on_domain(model, train_loader):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, (x, y) in enumerate(loop):
            x, y = x.to(device), y.squeeze().long().to(device) # Convert labels from shape [B, 1] to [B].

            # Ensure y is at least one-dimensional for batch operations.
            if y.ndim == 0:
                y = y.unsqueeze(0)

            # Skip empty batches if any appear after tensor reshaping.
            if y.size(0) == 0:
                continue

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(batch=batch_idx+1, loss=loss.item())
        print(f"  Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

# Evaluation
def evaluate(model, loader, name=""):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.squeeze().long().to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    accuracy = 100 * correct / total
    print(f"{name:<12} Accuracy: {accuracy:.2f}%")
    return accuracy

## 8. Experiment Blocks

Run the following sections to reproduce the baseline and PneumoNet experiments.

For faster debugging, reduce `num_epochs` or `num_runs`. For paper-style reporting, keep `num_runs = 3` and average the final accuracy and forgetting across runs.


### 8.1 Joint Training Baseline

Offline reference trained with all domains available simultaneously. This is not a continual learning setting, but it provides an upper reference.

In [ ]:
# Joint training baseline
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running Joint-training - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Reset hyperparameters and model for each run.
    lr = 0.001
    num_epochs = 50
    batch_size = 32

    all_train_datasets = []   # Merge all domain-specific training sets for offline joint training.

    model = Tiny_64_CNN().to(device)
    # Optional ablations:
    # model = Tiny_32_CNN().to(device)
    # model = CNN().to(device)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("method: PneumoNet CNN + Joint-training")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} MedMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        # merge
        all_train_datasets.append(train_data)
        test_loader = DataLoader(test_data, batch_size=batch_size)
        seen_test_sets.append((name, test_loader))

        # merge all domain data
        joint_train_data = ConcatDataset(all_train_datasets)

    # Train once using the concatenated multi-domain dataset.
    train_loader = DataLoader(joint_train_data, batch_size=batch_size, shuffle=True)
    train_on_domain(model, train_loader)

    print("== Evaluation on all seen domains ==")
    for test_name, loader in seen_test_sets:
        acc = evaluate(model, loader, test_name)
        domain_accuracies[test_name].append(acc) # Store the accuracy for the domain

    # Compute final average accuracy and average forgetting for this run.
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# Aggregate results across runs.
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

### 8.2 Fine-Tuning Baseline

Naive sequential training without replay. This baseline is expected to suffer from catastrophic forgetting.

In [ ]:
# Fine-tuning baseline
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running Fine-tuning - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Reset hyperparameters and model for each run.
    lr = 0.001
    num_epochs = 50
    batch_size = 32

    model = Tiny_64_CNN().to(device)
    # Optional ablations:
    # model = Tiny_32_CNN().to(device)
    # model = CNN().to(device)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("method: PneumoNet CNN + Fine-tuning")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        # Train sequentially on the current domain.
        train_on_domain(model, train_loader)
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    # Compute final average accuracy and average forgetting for this run.
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# Aggregate results across runs.
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

### 8.3 Experience Replay Baseline

Replay-based continual learning using standard reservoir sampling.

In [ ]:
# Experience Replay baseline
import time
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running ER - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Reset hyperparameters and model for each run.
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 500
    replay_ratio = 1.0

    model = Tiny_64_CNN().to(device)
    # Optional ablations:
    # model = Tiny_32_CNN().to(device)
    # model = CNN().to(device)
    buffer = ReservoirBuffer(buffer_capacity)
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: PneumoNet CNN + ER")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        # Train sequentially on the current domain.
        # start_time = time.time()
        train_with_er(model, train_loader, buffer)
        # end_time = time.time()
        # print(f"Training time: {end_time - start_time:.2f} seconds")
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            # start_time = time.time()
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)
            # end_time = time.time()
            # print(f"Evaluation time: {end_time - start_time:.2f} seconds")

    # Compute final average accuracy and average forgetting for this run.
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# Aggregate results across runs.
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

### 8.4 Class-Balancing Reservoir Sampling Baseline

Replay-based continual learning with class-balanced reservoir storage.

In [ ]:
# Class-Balancing Reservoir Sampling baseline
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running CBRS - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Reset hyperparameters and model for each run.
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 500
    replay_ratio = 1.0

    model = Tiny_64_CNN().to(device)
    # Optional ablations:
    # model = Tiny_32_CNN().to(device)
    # model = CNN().to(device)

    buffer = CBRSBuffer(capacity_per_class=250, num_classes=2)  # total buffer capacity = 500
    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: PneumoNet CNN + CBRS")

    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        train_with_er(model, train_loader, buffer)
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            acc = evaluate(model, loader, test_name)
            domain_accuracies[test_name].append(acc)

    # Compute final average accuracy and average forgetting for this run.
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# Aggregate results across runs.
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")

## 9. Efficiency Utilities

The following helper functions compute model size, memory usage, FLOPs, and trainable parameter count for on-device efficiency analysis.

In [ ]:
!pip install -q torchinfo==1.8.0
!pip install -q fvcore
from fvcore.nn import FlopCountAnalysis
from torchinfo import summary
import time

# Model efficiency utilities
# Function to estimate serialized model size.
def get_model_size(model):
    torch.save(model.state_dict(), "model.pt")
    size_mb = os.path.getsize("model.pt") / (1024 * 1024)
    os.remove("model.pt")
    return size_mb

"""
# Function to get inference time
def get_inference_time(model, dataloader, device):
    model.eval()
    start_time = time.time()
    with torch.no_grad():
        for inputs, _ in dataloader:
            inputs = inputs.to(device)
            _ = model(inputs)
    end_time = time.time()
    return (end_time - start_time) * 1000 / len(dataloader) # ms per batch
"""

# Estimate parameter-memory usage during inference.
def get_memory_usage(model, input_shape):
    # This gives a rough estimate. Detailed profiling may require hardware-specific tools.
    return summary(model, input_shape=input_shape, verbose=0).total_param_bytes / (1024 * 1024)

# Estimate FLOPs for a single forward pass.
def get_flops(model, input_shape):
    # Create a dummy input tensor with the specified shape.
    dummy_input = torch.randn(input_shape)
    # Move the dummy input to the same device as the model.
    dummy_input = dummy_input.to(next(model.parameters()).device)

    # Calculate FLOPs.
    flops = FlopCountAnalysis(model, dummy_input)
    return flops.total() / 1e6 # Return in MFLOPs

# Count trainable parameters.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 10. Proposed Method: PneumoNet

This block runs the paper-default PneumoNet configuration:

- PneumoNet lightweight CNN (`Tiny_64_CNN`)
- dual-stage balanced buffer with total capacity 500
- replay ratio 1.0
- dynamic class-weighted loss
- 3 runs for mean and standard deviation reporting


In [ ]:
############################################
# Proposed method: PneumoNet
num_runs = 3
all_run_results = []

for run_id in range(num_runs):
    print(f"\n=== Running PneumoNet - Run {run_id + 1} ===")
    torch.manual_seed(run_id)
    random.seed(run_id)
    np.random.seed(run_id)
    print(f"Using seed: {run_id}")

    # Reset hyperparameters and model for each run.
    lr = 0.001
    num_epochs = 50
    batch_size = 32
    buffer_capacity = 500
    replay_ratio = 1.0

    model = Tiny_64_CNN().to(device)
    # Optional model ablations:
    # model = Tiny_32_CNN().to(device)
    # model = CNN().to(device)
    # model = MobileNetV2(num_classes=2).to(device)
    # model = MobileNetV2_Pretrained(num_classes=2).to(device)
    buffer = DualStageBuffer(capacity_per_class=250, num_classes=2)

    seen_test_sets = []
    domain_accuracies = defaultdict(list)

    print("dataset: PneumoniaMNIST")
    print("num_epochs:", num_epochs)
    print("replay_ratio:", replay_ratio)
    print("buffer_capacity:", buffer_capacity)
    print("method: PneumoNet")

    model_size = get_model_size(model)
    memory_use = get_memory_usage(model, input_shape=(batch_size, 1, 28, 28))
    flops_values = get_flops(model, input_shape=(batch_size, 1, 28, 28))
    param_count = count_parameters(model)
    print(f'PneumoNet model size: {model_size:.4f} MB')
    print(f'PneumoNet memory usage estimate: {memory_use:.4f} MB')
    print(f'PneumoNet FLOPs: {flops_values:.2f} MFLOPs')
    print(f'PneumoNet parameter count: {param_count}')


    for name, transform in zip(domain_names, domain_transforms):
        print(f"\n== Training on {name} PneumoniaMNIST ==")
        train_data = DataClass(split='train', transform=transform, download=True, root=dataset_root)
        test_data = DataClass(split='test', transform=transform, download=True, root=dataset_root)
        print(f"Number of training samples: {len(train_data)}, Number of test samples: {len(test_data)}")

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=batch_size)

        # Train with dual-stage replay and dynamic class-weighted loss.
        start_time = time.time()
        train_with_er_dynamic_weight(model, train_loader, buffer)
        end_time = time.time()
        print(f"Training time: {end_time - start_time:.2f} seconds")
        seen_test_sets.append((name, test_loader))

        print("== Evaluation on all seen domains ==")
        for test_name, loader in seen_test_sets:
            start_time = time.time()
            acc = evaluate(model, loader, test_name)
            end_time = time.time()
            print(f"Inference time: {end_time - start_time:.2f} seconds")
            domain_accuracies[test_name].append(acc)

    # Compute final average accuracy and average forgetting for this run.
    print(f"\n=== Results for Run {run_id + 1} ===")
    run_forgetting = {}
    total_forgetting_run = 0
    for domain in domain_accuracies:
        if len(domain_accuracies[domain]) > 1:
            max_acc = max(domain_accuracies[domain][:-1])
            last_acc = domain_accuracies[domain][-1]
            forgetting = max_acc - last_acc
        else:
            forgetting = 0.0
        run_forgetting[domain] = forgetting
        total_forgetting_run += forgetting
        print(f"{domain:<12}: Forgetting = {forgetting:.2f}%")

    avg_acc_run = sum([accs[-1] for accs in domain_accuracies.values()]) / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0
    avg_forgetting_run = total_forgetting_run / len(domain_accuracies) if len(domain_accuracies) > 0 else 0.0

    print(f"\nRun {run_id + 1} Final Average Accuracy: {avg_acc_run:.2f}%")
    print(f"Run {run_id + 1} Average Forgetting: {avg_forgetting_run:.2f}%")

    all_run_results.append({
        'run_id': run_id,
        'domain_accuracies': domain_accuracies,
        'avg_accuracy': avg_acc_run,
        'avg_forgetting': avg_forgetting_run
    })

# Aggregate results across runs.
print("\n=== Average Results Across All Runs ===")
if all_run_results:
    avg_final_accuracies = [run['avg_accuracy'] for run in all_run_results]
    avg_forgettings = [run['avg_forgetting'] for run in all_run_results]

    mean_avg_accuracy = np.mean(avg_final_accuracies)
    mean_avg_forgetting = np.mean(avg_forgettings)
    std_avg_accuracy = np.std(avg_final_accuracies)
    std_avg_forgetting = np.std(avg_forgettings)

    print(f"Mean Final Average Accuracy: {mean_avg_accuracy:.2f}% (Std: {std_avg_accuracy:.2f})")
    print(f"Mean Average Forgetting: {mean_avg_forgetting:.2f}% (Std: {std_avg_forgetting:.2f})")

else:
    print("No run results were recorded.")
